In [1]:
import pandas as pd
import numpy as np
import os
import json
os.chdir('..')

In [2]:
from src.utils.eval_utils import calculate_metrics

In [3]:
import re

In [36]:
prompt_type = 'legoabsa'
label = '[A] kamarnya [O] butuh perbaikan [S] negative [SSEP] [A] dinding kamar [O] butuh perbaikan [S] negative [SSEP] [A] pintupintu [O] butuh perbaikan [S] negative'
pred = '[A]       kamarnya [O]butuh perbaikan[S]negative[SSEP][A] dinding kamar [O] butuh perbaikan [S] negative [SSEP][A] pintupintu [O]butuh perbaikan[S] negative'
label = '( fasilitas | biasanya ada | positive ) ; ( cemilan | hanya setengah dari pada biasanya | negative ) ; ( airy room | sangat sering menggunakan | positive )'
pred = '(fasilitas|biasanya ada|positive ) ; (cemilan |hanya setengah dari pada biasanya|negative);( airy room | sangat sering menggunakan | positive )'
label = '<|aspect|> kamarnya <|opinion|> butuh perbaikan <|sentiment|> negative;<|aspect|> dinding kamar <|opinion|> butuh perbaikan <|sentiment|> negative;<|aspect|> pintupintu <|opinion|> butuh perbaikan <|sentiment|> negative'
pred = '<|aspect|>     kamarnya <|opinion|>butuh perbaikan <|sentiment|>negative;<|aspect|>dinding kamar <|opinion|>        butuh perbaikan <|sentiment|> negative;<|aspect|>pintupintu       <|opinion|> butuh perbaikan<|sentiment|> negative'

In [37]:
if prompt_type == "mvp":
	# Postprocess the prediction for MvP
	# Split the target and prediction into lists
	target_split = label.split("[SSEP]")
	pred_split = pred.split("[SSEP]")
	# Strip whitespace
	target_split = [l.strip() for l in target_split]
	pred_split = [l.strip() for l in pred_split]

	# Make sure there is a whitespace before and after [A], [O], and [S]
	target_split = [l.replace("[A]", " [A] ").replace("[O]", " [O] ").replace("[S]", " [S] ").strip() for l in target_split]
	pred_split = [l.replace("[A]", " [A] ").replace("[O]", " [O] ").replace("[S]", " [S] ").strip() for l in pred_split]

	# Replace double or more whitespaces with a single whitespace
	target_split = [re.sub(r'\s+', ' ', l) for l in target_split]
	pred_split = [re.sub(r'\s+', ' ', l) for l in pred_split]

elif prompt_type == "gas":
	# Split the target and prediction into lists (GAS and LegoABSA)
	target_split = label.split(';')
	pred_split = pred.split(';')
	target_split = [l.strip() for l in target_split]
	pred_split = [l.strip() for l in pred_split]

	# Make sure there is a whitespace before and after '(' and '|'
	target_split = [l.replace('(', ' ( ').replace(')', ' ) ').replace('|', ' | ').strip() for l in target_split]
	pred_split = [l.replace('(', ' ( ').replace(')', ' ) ').replace('|', ' | ').strip() for l in pred_split]

	# Replace double or more whitespaces with a single whitespace
	target_split = [re.sub(r'\s+', ' ', l) for l in target_split]
	pred_split = [re.sub(r'\s+', ' ', l) for l in pred_split]

elif prompt_type == "legoabsa":
	# Split the target and prediction into lists (GAS and LegoABSA)
	target_split = label.split(';')
	pred_split = pred.split(';')
	target_split = [l.strip() for l in target_split]
	pred_split = [l.strip() for l in pred_split]

	# Make sure there is a whitespace before and after '(' and '|'
	target_split = [l.replace('<|aspect|>', ' <|aspect|> ').replace('<|opinion|>', ' <|opinion|> ').replace('<|sentiment|>', ' <|sentiment|> ').strip() for l in target_split]
	pred_split = [l.replace('<|aspect|>', ' <|aspect|> ').replace('<|opinion|>', ' <|opinion|> ').replace('<|sentiment|>', ' <|sentiment|> ').strip() for l in pred_split]

	# Replace double or more whitespaces with a single whitespace
	target_split = [re.sub(r'\s+', ' ', l) for l in target_split]
	pred_split = [re.sub(r'\s+', ' ', l) for l in pred_split]
	
else:
	raise ValueError(f"Unknown prompt type: {prompt_type}")

In [38]:
print("Target:", target_split)
print("Pred:", pred_split)
print(target_split == pred_split)

Target: ['<|aspect|> kamarnya <|opinion|> butuh perbaikan <|sentiment|> negative', '<|aspect|> dinding kamar <|opinion|> butuh perbaikan <|sentiment|> negative', '<|aspect|> pintupintu <|opinion|> butuh perbaikan <|sentiment|> negative']
Pred: ['<|aspect|> kamarnya <|opinion|> butuh perbaikan <|sentiment|> negative', '<|aspect|> dinding kamar <|opinion|> butuh perbaikan <|sentiment|> negative', '<|aspect|> pintupintu <|opinion|> butuh perbaikan <|sentiment|> negative']
True
